##### Ingest results.json file

##### Read the JSON file using the spark dataframe reader API

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2021-03-28")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
results_schema = StructType(fields=[StructField("resultId", IntegerType(), False),
                                    StructField("raceId", IntegerType(), True),
                                    StructField("driverId", IntegerType(), True),
                                    StructField("constructorId", IntegerType(), True),
                                    StructField("number", IntegerType(), True),
                                    StructField("grid", IntegerType(), True),
                                    StructField("position", IntegerType(), True),
                                    StructField("positionText", StringType(), True),
                                    StructField("positionOrder", IntegerType(), True),
                                    StructField("points", FloatType(), True),
                                    StructField("laps", IntegerType(), True),
                                    StructField("time", StringType(), True),
                                    StructField("milliseconds", IntegerType(), True),
                                    StructField("fastestLap", IntegerType(), True),
                                    StructField("rank", IntegerType(), True),
                                    StructField("fastestLapTime", StringType(), True),
                                    StructField("fastestLapSpeed", FloatType(), True),
                                    StructField("statusId", StringType(), True)])

In [0]:
results_df = spark.read \
.schema(results_schema) \
.json(f"{raw_path}/{v_file_date}/results.json")

###### Rename columns and add new columns

In [0]:
results_with_columns_df = results_df.withColumnRenamed("resultId", "result_id") \
                                    .withColumnRenamed("raceId", "race_id") \
                                    .withColumnRenamed("driverId", "driver_id") \
                                    .withColumnRenamed("constructorId", "constructor_id") \
                                    .withColumnRenamed("positionText", "position_text") \
                                    .withColumnRenamed("positionOrder", "position_order") \
                                    .withColumnRenamed("fastestLap", "fastest_lap") \
                                    .withColumnRenamed("fastestLapTime", "fastest_lap_time") \
                                    .withColumnRenamed("fastestLapSpeed", "fastest_lap_speed") \
                                    .withColumn("ingestion_date", current_timestamp()) \
                                    .withColumn("data_source", lit(v_data_source)) \
                                    .withColumn("file_date", lit(v_file_date))

In [0]:
results_final_df = add_ingestion_date(results_with_columns_df)

##### Drop unwanted column

In [0]:
results_final_df = results_with_columns_df.drop(col("statusId"))

##### Write to output to processed container in parquet format

In [0]:
results_final_df.write.mode("overwrite").partitionBy('race_id').parquet(f"{processed_path}/results")

###### Implement incremental load

###### Method 1

In [0]:
# for race_id_list in results_final_df.select("race_id").distinct().collect():
#   if (spark._jsparkSession.catalog().tableExists("f1_processed.results")):
#     spark.sql(f"ALTER TABLE f1_processed.results DROP IF EXISTS PARTITION (race_id = {race_id_list.race_id})")

###### Solution for delta tables

In [0]:
if spark.catalog.tableExists("f1_processed.results"):
    race_ids = [row.race_id for row in results_final_df.select("race_id").distinct().collect()]
    spark.sql(f"""
    DELETE FROM f1_processed.results
    WHERE race_id IN ({','.join(map(str, race_ids))})
    """)
else:
    print("Table f1_processed.results does not exist. Skipping DELETE.")

In [0]:
# results_final_df.write.mode("append").format("delta").saveAsTable(f"f1_processed.results")

###### Method 2

In [0]:
### final solution 
overwrite_partition_free(results_final_df, 'f1_processed', 'results', 'race_id')

In [0]:
# results_final_df.write.mode("append").format("delta").saveAsTable(f"f1_processed.results")

##### Check

In [0]:
# spark.read.parquet(f"{processed_path}/results").display()

In [0]:
dbutils.notebook.exit("SUCCESS")

In [0]:
%sql
select race_id, count(1) as count
from f1_processed.results
group by race_id
order by race_id desc;